# Day 2 — Notebook 1: Azure OpenAI Setup & First API Call
**Tax Tech Transformation Program | Big4 Firm**

This notebook covers:
- Installing required libraries
- Configuring Azure OpenAI credentials securely
- Making your first chat completion call
- Free-tier alternative (Groq) setup

⚠️ **Never hard-code API keys. Always use `getpass` or environment variables.**

## Step 1: Install Required Libraries

In [1]:
# Run this cell first — installs all required packages
%pip install openai python-dotenv groq --quiet
print("✅ Libraries installed successfully")


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
✅ Libraries installed successfully


## Step 2: Configure Azure OpenAI Credentials

You need three values from your Azure OpenAI resource:
1. **Endpoint** — e.g. `https://YOUR-RESOURCE.openai.azure.com/`
2. **API Key** — from Azure Portal → Keys & Endpoints
3. **Deployment Name** — the name you gave when deploying the model (e.g. `gpt-4o`, `gpt-35-turbo`)

> **Where to find them:** Azure Portal → Azure OpenAI → Your Resource → Keys and Endpoint

In [2]:
import os
from getpass import getpass
from openai import AzureOpenAI

In [ ]:
# Securely collect credentials — input is hidden
AZURE_OPENAI_ENDPOINT   = input("Enter your Azure OpenAI Endpoint (e.g. https://xxx.openai.azure.com/): ").strip()
AZURE_OPENAI_API_KEY    = getpass("Enter your Azure OpenAI API Key: ")
AZURE_DEPLOYMENT_NAME   = input("Enter your Deployment Name (e.g. gpt-4o or gpt-35-turbo): ").strip()
AZURE_API_VERSION       = "2024-02-01"   # Stable GA version — change if your resource uses a different one

# Initialise client
client = AzureOpenAI(
    azure_endpoint = AZURE_OPENAI_ENDPOINT,
    api_key        = AZURE_OPENAI_API_KEY,
    api_version    = AZURE_API_VERSION,
)
print("✅ Azure OpenAI client initialised successfully!")

Alternatively, you can uncomment and run the following section to use .env variables.

In [3]:
# from dotenv import load_dotenv

# load_dotenv()

# AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
# AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
# AZURE_DEPLOYMENT_NAME = os.getenv("AZURE_DEPLOYMENT_NAME")
# AZURE_API_VERSION = os.getenv("AZURE_API_VERSION")

# # Initialise client
# client = AzureOpenAI(
#     azure_endpoint = AZURE_OPENAI_ENDPOINT,
#     api_key        = AZURE_OPENAI_API_KEY,
#     api_version    = AZURE_API_VERSION,
# )
# print("✅ Azure OpenAI client initialised successfully!")

✅ Azure OpenAI client initialised successfully!


## Step 3: Test the Connection — First API Call

In [4]:
# Simple test call — should return a short tax answer
response = client.chat.completions.create(
    model    = AZURE_DEPLOYMENT_NAME,
    messages = [
        {"role": "system", "content": "You are a concise tax assistant for Big4 professionals."},
        {"role": "user",   "content": "In one sentence, what is GST?"}
    ],
    temperature = 0.3,
    max_tokens  = 80,
)

answer = response.choices[0].message.content
print("=" * 60)
print("Model response:")
print(answer)
print("=" * 60)
print(f"Model used    : {response.model}")
print(f"Prompt tokens : {response.usage.prompt_tokens}")
print(f"Completion    : {response.usage.completion_tokens}")
print(f"Total tokens  : {response.usage.total_tokens}")

Model response:
GST (Goods and Services Tax) is a comprehensive, multi-stage, destination-based indirect tax levied on the supply of goods and services in India, replacing multiple indirect taxes.
Model used    : gpt-4o-2024-11-20
Prompt tokens : 30
Completion    : 36
Total tokens  : 66


## Step 4: Understanding API Parameters

| Parameter | What it does | Tax context |
|-----------|-------------|-------------|
| `temperature=0` | Fully deterministic | Use for compliance / rules |
| `temperature=0.3` | Mostly deterministic | Use for tax Q&A |
| `temperature=0.7` | More creative | Use for drafting |
| `max_tokens` | Caps response length | Control cost |
| `system` message | Sets the agent persona | Critical for tax accuracy |

In [5]:
# Demonstrate temperature effect
for temp in [0.0, 0.5, 1.0]:
    r = client.chat.completions.create(
        model       = AZURE_DEPLOYMENT_NAME,
        messages    = [
            {"role": "system", "content": "You are a tax assistant."},
            {"role": "user",   "content": "Name one benefit of tax loss harvesting."}
        ],
        temperature = temp,
        max_tokens  = 60,
    )
    print(f"Temperature {temp}: {r.choices[0].message.content.strip()}")
    print("-" * 50)

Temperature 0.0: One key benefit of tax loss harvesting is **reducing taxable income**. By selling investments that have declined in value and realizing a capital loss, taxpayers can offset capital gains from other investments, thereby lowering their overall tax liability. If losses exceed gains, up to $3,000 of the excess loss
--------------------------------------------------
Temperature 0.5: One key benefit of tax loss harvesting is **reducing taxable income**. By selling investments at a loss, taxpayers can offset capital gains from profitable investments or, if losses exceed gains, deduct up to $3,000 ($1,500 if married filing separately) from ordinary income in a tax year
--------------------------------------------------
Temperature 1.0: One benefit of tax loss harvesting is that it can help reduce your taxable income. By selling investments that have decreased in value and realizing a capital loss, you can use that loss to offset capital gains from other investments or even up 

## Step 5: Exercise 1 — Prompt Engineering (10 min)

Modify the system prompt and user message below to get answers for:
1. What is the TDS rate for contractor payments in India?
2. Explain Section 80C deductions in under 50 words.
3. What documents are needed for transfer pricing compliance?

Observe how the **system prompt** changes the tone and format of answers.

In [6]:
# ── YOUR EXERCISE CODE HERE ──────────────────────────────────────────────────
# Task: Change system_prompt and user_question below and re-run this cell

system_prompt = "You are a senior tax advisor at a Big4 firm in India. Answer precisely."
user_question  = "What is TDS rate for contractor payments?"

# ─────────────────────────────────────────────────────────────────────────────
response = client.chat.completions.create(
    model       = AZURE_DEPLOYMENT_NAME,
    messages    = [
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": user_question},
    ],
    temperature = 0.2,
    max_tokens  = 200,
)
print(response.choices[0].message.content)

The TDS (Tax Deducted at Source) rate for contractor payments under Section 194C of the Income Tax Act is as follows:

1. **Individual/HUF (Hindu Undivided Family)**: **1%**
2. **Other entities (e.g., companies, firms, etc.)**: **2%**

**Key Points:**
- TDS is applicable if the payment exceeds ₹30,000 for a single contract or ₹1,00,000 in aggregate during the financial year.
- No TDS is required if the contractor provides a declaration under Section 197 for lower or nil deduction of tax.
- The contractor must furnish their PAN; otherwise, TDS will be deducted at a higher rate of **20%**.




## Optional: Groq Free-Tier Alternative

If Azure keys are unavailable, use Groq (free, very fast).
1. Sign up at https://console.groq.com
2. Create an API key
3. Run the cell below — same code structure as Azure OpenAI

In [7]:
# Groq alternative — only run if you don't have Azure access
# pip install groq   (already installed above)

from groq import Groq
from getpass import getpass

GROQ_API_KEY = getpass("Enter your Groq API Key: ")
groq_client  = Groq(api_key=GROQ_API_KEY)

response = groq_client.chat.completions.create(
    model    = "llama-3.3-70b-versatile",   # Free model on Groq
    messages = [
        {"role": "system", "content": "You are a tax assistant."},
        {"role": "user",   "content": "What is transfer pricing in one sentence?"},
    ],
    temperature = 0.2,
    max_tokens  = 100,
)
print("Groq Response:", response.choices[0].message.content)

Enter your Groq API Key:  ········


Groq Response: Transfer pricing refers to the pricing of goods, services, and intangibles that are transferred between related entities, such as subsidiaries or affiliates, within a multinational corporation, and is subject to taxation and regulatory scrutiny to ensure fairness and compliance with tax laws.
